# FinOptix — Algorithmic Trading & Backtesting

### Objective
To Build a rules-based technical trading strategy and evaluate it using a **Walk-Forward Optimization (WFO)** pipeline across multiple crypto assets and market regimes.


In [20]:
import warnings
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from dateutil.relativedelta import relativedelta

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [21]:

TICKERS = ["BTC-USD", "ETH-USD", "LTC-USD", "XRP-USD", "BCH-USD"]
RISK_FREE_TICKER = "^TNX"

START_DATE = "2018-01-01"
END_DATE   = "2023-01-01"

INITIAL_CAPITAL      = 1_000_000
TRANSACTION_COST_PCT = 0.001
RISK_PER_TRADE       = 0.01
MAX_TRADE_DRAWDOWN   = 0.25
TRADING_DAYS_PER_YEAR = 252

WFO_GRID = {
    'adx_threshold': [20, 25, 30],
    'stop_loss_pct': [0.10, 0.15],
    'take_profit_pct': [0.15, 0.20],
    'atr_trail_mult': [5, 7, 9]
}

In [22]:
def get_price_data(ticker, start, end):
    df = yf.download(ticker, start=start, end=end, progress=False)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df = df.reset_index()
    df = df[["Date", "Open", "High", "Low", "Close", "Volume"]].dropna().reset_index(drop=True)
    return df

def add_regime_filters(df):
    out = df.copy()
    out['SMA_200'] = out['Close'].rolling(window=200).mean()
    out['Trend_Regime'] = np.where(out['Close'] > out['SMA_200'], 'Bull', 'Bear')

    out['Daily_Return'] = out['Close'].pct_change()
    out['Rolling_Vol'] = out['Daily_Return'].rolling(window=30).std()
    out['Vol_Regime'] = pd.qcut(out['Rolling_Vol'].rank(method='first'), q=3, labels=['Low', 'Medium', 'High'])
    return out

market_data = {}
for t in TICKERS:
    raw = get_price_data(t, START_DATE, END_DATE)
    market_data[t] = add_regime_filters(raw)
    print(f"{t}: {len(raw)} rows fetched.")

rf_raw = get_price_data(RISK_FREE_TICKER, START_DATE, END_DATE)

BTC-USD: 1826 rows fetched.
ETH-USD: 1826 rows fetched.
LTC-USD: 1826 rows fetched.
XRP-USD: 1826 rows fetched.
BCH-USD: 1826 rows fetched.


In [23]:
def true_range(df):
    prev_close = df["Close"].shift(1)
    ranges = pd.concat([
        df["High"] - df["Low"],
        (df["High"] - prev_close).abs(),
        (df["Low"] - prev_close).abs(),
    ], axis=1)
    return ranges.max(axis=1)

def atr(df, period=14):
    return true_range(df).rolling(period).mean()

def macd(close, fast=12, slow=26, signal=9):
    ema_fast = close.ewm(span=fast, adjust=False).mean()
    ema_slow = close.ewm(span=slow, adjust=False).mean()
    macd_line = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal, adjust=False).mean()
    return macd_line, signal_line

def adx(df, period=14):
    up_move = df["High"].diff()
    down_move = -df["Low"].diff()
    plus_dm = np.where((up_move > down_move) & (up_move > 0), up_move, 0.0)
    minus_dm = np.where((down_move > up_move) & (down_move > 0), down_move, 0.0)
    tr_avg = true_range(df).rolling(period).mean()
    plus_di = 100 * pd.Series(plus_dm, index=df.index).rolling(period).mean() / tr_avg
    minus_di = 100 * pd.Series(minus_dm, index=df.index).rolling(period).mean() / tr_avg
    dx = 100 * (plus_di - minus_di).abs() / (plus_di + minus_di)
    adx_line = dx.rolling(period).mean()
    return plus_di, minus_di, adx_line

def generate_combined_signals(df, adx_threshold=20):
    out = df.copy()
    macd_line, signal_line = macd(out["Close"])
    plus_di, minus_di, adx_line = adx(out)
    out["ATR"] = atr(out)

    long_e = (plus_di > minus_di) & (plus_di.shift(1) <= minus_di.shift(1)) & (adx_line > adx_threshold) & (macd_line > signal_line)
    short_e = (minus_di > plus_di) & (minus_di.shift(1) <= plus_di.shift(1)) & (adx_line > adx_threshold) & (macd_line < signal_line)

    out["Signal"] = 0
    out.loc[long_e, "Signal"] = 1
    out.loc[short_e, "Signal"] = -1
    return out

In [24]:
def run_backtest(df, initial_capital=INITIAL_CAPITAL, stop_loss_pct=0.10, take_profit_pct=0.20,
                 atr_trail_mult=7, max_trade_drawdown=MAX_TRADE_DRAWDOWN,
                 transaction_cost_pct=TRANSACTION_COST_PCT, risk_per_trade=RISK_PER_TRADE):

    dates = df["Date"].to_numpy()
    close = df["Close"].to_numpy(dtype=float)
    high = df["High"].to_numpy(dtype=float)
    low = df["Low"].to_numpy(dtype=float)
    signal = df["Signal"].fillna(0).to_numpy(dtype=int)
    atr_vals = df["ATR"].to_numpy(dtype=float)
    n = len(df)

    cash = float(initial_capital)
    position = 0
    entry_price = entry_date = None
    entry_idx = 0
    shares = 0.0
    trail_stop = trail_tp = np.nan
    trade_peak_value = cash

    portfolio_value = np.zeros(n)
    trades = []

    def open_position(i, direction, price, capital, atr_val):
        stop_distance = atr_trail_mult * atr_val if not np.isnan(atr_val) else price * stop_loss_pct
        risk_amount = capital * risk_per_trade
        target_shares = risk_amount / stop_distance if stop_distance > 0 else 0

        max_usable_capital = capital * (1 - transaction_cost_pct)
        max_shares = max_usable_capital / price
        shares = min(target_shares, max_shares)

        remaining = capital - (shares * price * transaction_cost_pct)

        if direction == 1:
            ts = price - stop_distance
            tp = price * (1 + take_profit_pct)
        else:
            ts = price + stop_distance
            tp = price * (1 - take_profit_pct)
        return remaining, shares, ts, tp

    for i in range(n):
        price = close[i]
        if position == 0:
            portfolio_value[i] = cash
            if signal[i] != 0 and not np.isnan(atr_vals[i]):
                position = int(signal[i])
                entry_price, entry_date, entry_idx = price, dates[i], i
                cash, shares, trail_stop, trail_tp = open_position(i, position, price, cash, atr_vals[i])
                trade_peak_value = cash
            continue

        if position == 1:
            mtm_value = cash + (shares * price)
        else:
            mtm_value = cash + shares * (entry_price - price)

        portfolio_value[i] = mtm_value
        trade_peak_value = max(trade_peak_value, mtm_value)
        trade_dd = (trade_peak_value - mtm_value) / trade_peak_value if trade_peak_value > 0 else 0.0

        if not np.isnan(atr_vals[i]):
            if position == 1:
                trail_stop = max(trail_stop, price - atr_trail_mult * atr_vals[i])
            else:
                trail_stop = min(trail_stop, price + atr_trail_mult * atr_vals[i])

        exit_reason = None
        if position == 1:
            if low[i] <= trail_stop: exit_reason = "stop_loss"
            elif high[i] >= trail_tp: exit_reason = "take_profit"
        else:
            if high[i] >= trail_stop: exit_reason = "stop_loss"
            elif low[i] <= trail_tp: exit_reason = "take_profit"

        if trade_dd >= max_trade_drawdown: exit_reason = "max_drawdown"
        if exit_reason is None and signal[i] == -position: exit_reason = "signal_reversal"

        if exit_reason:
            exit_cost = (shares * price) * transaction_cost_pct
            cash_after_exit = mtm_value - exit_cost
            trades.append({
                "entry_date": entry_date, "exit_date": dates[i],
                "direction": "long" if position == 1 else "short",
                "entry_price": entry_price, "exit_price": price,
                "holding_days": i - entry_idx, "exit_reason": exit_reason,
            })
            position = 0
            cash = portfolio_value[i] = cash_after_exit

    daily_df = pd.DataFrame({"Date": dates, "Close": close, "Portfolio_Value": portfolio_value})
    trades_df = pd.DataFrame(trades)
    return trades_df, daily_df

In [25]:
def compute_metrics(trades_df, daily_df, initial_capital=INITIAL_CAPITAL, periods_per_year=TRADING_DAYS_PER_YEAR):
    if daily_df.empty: return {}
    daily = daily_df.copy()
    daily["Return"] = daily["Portfolio_Value"].pct_change().fillna(0)
    total_return_pct = (daily["Portfolio_Value"].iloc[-1] / initial_capital - 1) * 100

    excess = daily["Return"] - 0.0
    sharpe = (excess.mean() / excess.std()) * np.sqrt(periods_per_year) if excess.std() > 0 else 0

    return {
        "Total Return (%)": round(total_return_pct, 2),
        "Trades": len(trades_df),
        "Sharpe": round(sharpe, 2)
    }

In [26]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def walk_forward_optimization(df, param_grid, train_months=12, test_months=3):
    dates = pd.to_datetime(df['Date'])
    start_date = dates.min()
    end_date = dates.max()

    keys, values = zip(*param_grid.items())
    param_combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]

    current_train_start = start_date
    oos_results = []
    all_oos_daily_returns = []
    all_oos_trades = []

    while True:
        train_end = current_train_start + relativedelta(months=train_months)
        test_end = train_end + relativedelta(months=test_months)
        if train_end >= end_date: break

        train_df = df[(dates >= current_train_start) & (dates < train_end)].copy()
        test_df = df[(dates >= train_end) & (dates < test_end)].copy()
        if test_df.empty: break

        best_sharpe, best_params = -999, None

        # Grid Search
        for params in param_combinations:
            sig_df = generate_combined_signals(train_df, adx_threshold=params['adx_threshold'])
            t_df, d_df = run_backtest(sig_df, stop_loss_pct=params['stop_loss_pct'],
                                      take_profit_pct=params['take_profit_pct'],
                                      atr_trail_mult=params['atr_trail_mult'])
            sharpe = compute_metrics(t_df, d_df).get("Sharpe", -999)
            if sharpe > best_sharpe:
                best_sharpe = sharpe
                best_params = params

        # Run Best Params Out-of-Sample
        if best_params is not None:
            sig_df_oos = generate_combined_signals(test_df, adx_threshold=best_params['adx_threshold'])
            t_df_oos, d_df_oos = run_backtest(
                sig_df_oos,
                stop_loss_pct=best_params['stop_loss_pct'],
                take_profit_pct=best_params['take_profit_pct'],
                atr_trail_mult=best_params['atr_trail_mult']
            )

            # Extract just the true OOS daily returns to stitch together
            d_df_oos['OOS_Return'] = d_df_oos['Portfolio_Value'].pct_change().fillna(0)
            d_df_oos['Benchmark_Return'] = d_df_oos['Close'].pct_change().fillna(0)

            # Slice strictly to the test window dates to avoid overlap
            oos_slice = d_df_oos[(pd.to_datetime(d_df_oos['Date']) >= pd.to_datetime(train_end.date())) &
                                 (pd.to_datetime(d_df_oos['Date']) < pd.to_datetime(test_end.date()))]

            all_oos_daily_returns.append(oos_slice[['Date', 'OOS_Return', 'Benchmark_Return']])
            if not t_df_oos.empty:
                all_oos_trades.append(t_df_oos)

            oos_metrics = compute_metrics(t_df_oos, d_df_oos)
            oos_results.append({
                'Test_Start': train_end.date(),
                'Test_End': test_end.date(),
                'IS_Sharpe': best_sharpe,
                'OOS_Sharpe': oos_metrics.get("Sharpe"),
                'OOS_Return': oos_metrics.get("Total Return (%)"),
                'Params': str(best_params)
            })

        current_train_start += relativedelta(months=test_months)

    # Stitch the continuous curve together
    if all_oos_daily_returns:
        master_daily = pd.concat(all_oos_daily_returns).set_index('Date')
        master_daily['Strat_Equity'] = INITIAL_CAPITAL * (1 + master_daily['OOS_Return']).cumprod()
        master_daily['Bench_Equity'] = INITIAL_CAPITAL * (1 + master_daily['Benchmark_Return']).cumprod()
        master_daily['Drawdown'] = (master_daily['Strat_Equity'] / master_daily['Strat_Equity'].cummax()) - 1
    else:
        master_daily = pd.DataFrame()

    master_trades = pd.concat(all_oos_trades, ignore_index=True) if all_oos_trades else pd.DataFrame()

    return pd.DataFrame(oos_results), master_daily, master_trades

In [27]:
# ---- Summary Results Printer ----
all_results = {}

for ticker in TICKERS:
    df = market_data[ticker]
    res_df, daily_df, trades_df = walk_forward_optimization(df, WFO_GRID)

    if daily_df.empty:
        continue

    # Core metrics
    strat_final = daily_df['Strat_Equity'].iloc[-1]
    bench_final = daily_df['Bench_Equity'].iloc[-1]
    total_return = (strat_final / INITIAL_CAPITAL - 1) * 100
    bench_return = (bench_final / INITIAL_CAPITAL - 1) * 100
    outperformance = total_return - bench_return

    # CAGR
    n_days = len(daily_df)
    n_years = n_days / TRADING_DAYS_PER_YEAR
    cagr = ((strat_final / INITIAL_CAPITAL) ** (1 / n_years) - 1) * 100

    # Max drawdown
    max_dd = daily_df['Drawdown'].min() * 100

    # Sharpe (OOS)
    oos_returns = daily_df['OOS_Return']
    sharpe = (oos_returns.mean() / oos_returns.std()) * np.sqrt(TRADING_DAYS_PER_YEAR) if oos_returns.std() > 0 else 0

    # Trade stats
    n_trades = len(trades_df)
    if not trades_df.empty:
        trades_df['Trade_Return_Pct'] = (trades_df['exit_price'] - trades_df['entry_price']) / trades_df['entry_price']
        trades_df.loc[trades_df['direction'] == 'short', 'Trade_Return_Pct'] *= -1
        win_rate = (trades_df['Trade_Return_Pct'] > 0).mean() * 100
    else:
        win_rate = 0

    all_results[ticker] = {
        'CAGR (%)': round(cagr, 2),
        'Total Return (%)': round(total_return, 2),
        'Benchmark Return (%)': round(bench_return, 2),
        'Outperformance (%)': round(outperformance, 2),
        'Sharpe (OOS)': round(sharpe, 2),
        'Max Drawdown (%)': round(max_dd, 2),
        'Total Trades': n_trades,
        'Win Rate (%)': round(win_rate, 2),
    }

# Print summary table
summary_df = pd.DataFrame(all_results).T
print("\n========== FINOPTIX — FULL OOS PERFORMANCE SUMMARY ==========")
print(summary_df.to_string())


========== FINOPTIX — FULL OOS PERFORMANCE SUMMARY ==========
         CAGR (%)  Total Return (%)  Benchmark Return (%)  Outperformance (%)  Sharpe (OOS)  Max Drawdown (%)  Total Trades  Win Rate (%)
BTC-USD      7.70             53.74                290.31             -236.57          1.20             -1.84         15.00         40.00
ETH-USD      7.58             52.76                652.17             -599.41          1.27             -2.54         20.00         40.00
LTC-USD      7.52             52.24                101.22              -48.98          1.35             -1.94         21.00         38.10
XRP-USD     12.88            101.85                -14.86              116.71          1.54             -1.50         25.00         52.00
BCH-USD      8.51             60.54                -45.23              105.77          1.34             -1.91         22.00         36.36


In [28]:
def plot_institutional_tearsheet(ticker, master_daily, master_trades, oos_summary):
    if master_daily.empty:
        print(f"No valid OOS data to plot for {ticker}")
        return

    # Create a 3-row Plotly figure
    fig = make_subplots(rows=3, cols=1,
                        shared_xaxes=False,
                        vertical_spacing=0.1,
                        subplot_titles=(f"{ticker}: Out-of-Sample Equity Curve vs Benchmark",
                                        "Strategy Drawdown Profile",
                                        "Distribution of Trade Returns (PnL)"),
                        row_heights=[0.5, 0.25, 0.25])

    # --- Plot 1: Equity Curve ---
    fig.add_trace(go.Scatter(x=master_daily.index, y=master_daily['Strat_Equity'],
                             mode='lines', name='WFO Strategy', line=dict(color='#00F0FF', width=2)),
                  row=1, col=1)
    fig.add_trace(go.Scatter(x=master_daily.index, y=master_daily['Bench_Equity'],
                             mode='lines', name='Buy & Hold Benchmark', line=dict(color='#888888', width=1, dash='dot')),
                  row=1, col=1)

    # --- Plot 2: Drawdown ---
    fig.add_trace(go.Scatter(x=master_daily.index, y=master_daily['Drawdown'],
                             mode='lines', name='Drawdown', fill='tozeroy',
                             line=dict(color='#FF3366', width=1), fillcolor='rgba(255, 51, 102, 0.3)'),
                  row=2, col=1)

    # --- Plot 3: Trade PnL Histogram ---
    if not master_trades.empty:
        # Calculate % return of each trade
        master_trades['Trade_Return_Pct'] = (master_trades['exit_price'] - master_trades['entry_price']) / master_trades['entry_price']
        master_trades.loc[master_trades['direction'] == 'short', 'Trade_Return_Pct'] *= -1

        fig.add_trace(go.Histogram(x=master_trades['Trade_Return_Pct'], nbinsx=40,
                                   name='Trade Returns', marker_color='#9966FF'),
                      row=3, col=1)

    # Styling
    fig.update_layout(height=900, template='plotly_dark',
                      title_text=f"Quantitative Tearsheet: {ticker} (Walk-Forward Out-of-Sample)",
                      hovermode="x unified")
    fig.update_yaxes(title_text="Portfolio Value ($)", row=1, col=1)
    fig.update_yaxes(title_text="Drawdown (%)", tickformat='.1%', row=2, col=1)
    fig.update_xaxes(title_text="Trade Return (%)", tickformat='.1%', row=3, col=1)

    fig.show()

    # Display the tabular WFO results
    print(f"--- WFO Rolling Fold Summary for {ticker} ---")
    display(oos_summary)

In [29]:
# Execute WFO and generate plots across all selected assets
wfo_summaries = {}

for ticker in TICKERS:
    print(f"\n========================================================")
    print(f"Analyzing {ticker}...")
    df = market_data[ticker]

    # Run the updated WFO engine
    res_df, daily_df, trades_df = walk_forward_optimization(df, WFO_GRID)
    wfo_summaries[ticker] = res_df

    # Generate the institutional visuals
    plot_institutional_tearsheet(ticker, daily_df, trades_df, res_df)


Analyzing BTC-USD...


--- WFO Rolling Fold Summary for BTC-USD ---


,Test_Start,Test_End,IS_Sharpe,OOS_Sharpe,OOS_Return,Params
0,2019-01-01,2019-04-01,2.21,-2.16,-0.77,"{'adx_threshold': 20, 'stop_loss_pct': 0.1, 't..."
1,2019-04-01,2019-07-01,2.08,-2.92,-0.67,"{'adx_threshold': 20, 'stop_loss_pct': 0.1, 't..."
2,2019-07-01,2019-10-01,1.90,-0.38,-0.11,"{'adx_threshold': 20, 'stop_loss_pct': 0.1, 't..."
3,2019-10-01,2020-01-01,1.61,1.95,5.18,"{'adx_threshold': 20, 'stop_loss_pct': 0.1, 't..."
4,2020-01-01,2020-04-01,1.18,2.27,3.02,"{'adx_threshold': 30, 'stop_loss_pct': 0.1, 't..."
5,2020-04-01,2020-07-01,1.66,0.93,0.21,"{'adx_threshold': 30, 'stop_loss_pct': 0.1, 't..."
6,2020-07-01,2020-10-01,1.68,1.63,5.64,"{'adx_threshold': 30, 'stop_loss_pct': 0.1, 't..."
7,2020-10-01,2021-01-01,1.73,0.00,0.00,"{'adx_threshold': 20, 'stop_loss_pct': 0.1, 't..."
8,2021-01-01,2021-04-01,1.44,1.58,2.40,"{'adx_threshold': 20, 'stop_loss_pct': 0.1, 't..."
9,2021-04-01,2021-07-01,1.32,1.73,3.42,"{'adx_threshold': 20, 'stop_loss_pct': 0.1, 't..."



Analyzing ETH-USD...


--- WFO Rolling Fold Summary for ETH-USD ---


...

--- WFO Rolling Fold Summary for XRP-USD ---


,Test_Start,Test_End,IS_Sharpe,OOS_Sharpe,OOS_Return,Params
0,2019-01-01,2019-04-01,1.94,1.56,4.35,"{'adx_threshold': 20, 'stop_loss_pct': 0.1, 't..."
1,2019-04-01,2019-07-01,1.82,2.01,7.99,"{'adx_threshold': 25, 'stop_loss_pct': 0.1, 't..."
2,2019-07-01,2019-10-01,1.50,2.35,13.21,"{'adx_threshold': 20, 'stop_loss_pct': 0.1, 't..."
3,2019-10-01,2020-01-01,2.01,1.42,3.42,"{'adx_threshold': 25, 'stop_loss_pct': 0.1, 't..."
4,2020-01-01,2020-04-01,1.70,2.07,2.01,"{'adx_threshold': 25, 'stop_loss_pct': 0.1, 't..."
5,2020-04-01,2020-07-01,2.18,2.17,12.80,"{'adx_threshold': 20, 'stop_loss_pct': 0.1, 't..."
6,2020-07-01,2020-10-01,2.21,1.75,4.75,"{'adx_threshold': 20, 'stop_loss_pct': 0.1, 't..."
7,2020-10-01,2021-01-01,2.28,2.02,5.88,"{'adx_threshold': 20, 'stop_loss_pct': 0.1, 't..."
8,2021-01-01,2021-04-01,2.22,-1.39,-0.40,"{'adx_threshold': 20, 'stop_loss_pct': 0.1, 't..."
9,2021-04-01,2021-07-01,2.08,1.81,2.40,"{'adx_threshold': 20, 'stop_loss_pct': 0.1, 't..."



Analyzing BCH-USD...


--- WFO Rolling Fold Summary for BCH-USD ---


,Test_Start,Test_End,IS_Sharpe,OOS_Sharpe,OOS_Return,Params
0,2019-01-01,2019-04-01,2.23,1.70,4.04,"{'adx_threshold': 20, 'stop_loss_pct': 0.1, 't..."
1,2019-04-01,2019-07-01,2.12,-2.88,-1.40,"{'adx_threshold': 20, 'stop_loss_pct': 0.1, 't..."
2,2019-07-01,2019-10-01,1.73,1.12,1.43,"{'adx_threshold': 20, 'stop_loss_pct': 0.1, 't..."
3,2019-10-01,2020-01-01,1.70,2.37,9.62,"{'adx_threshold': 20, 'stop_loss_pct': 0.1, 't..."
4,2020-01-01,2020-04-01,1.92,2.03,2.32,"{'adx_threshold': 20, 'stop_loss_pct': 0.1, 't..."
5,2020-04-01,2020-07-01,2.00,0.63,0.22,"{'adx_threshold': 25, 'stop_loss_pct': 0.1, 't..."
6,2020-07-01,2020-10-01,2.06,1.92,4.64,"{'adx_threshold': 20, 'stop_loss_pct': 0.1, 't..."
7,2020-10-01,2021-01-01,1.86,-2.38,-1.34,"{'adx_threshold': 20, 'stop_loss_pct': 0.1, 't..."
8,2021-01-01,2021-04-01,1.53,-0.42,-0.07,"{'adx_threshold': 30, 'stop_loss_pct': 0.1, 't..."
9,2021-04-01,2021-07-01,0.99,1.34,1.20,"{'adx_threshold': 30, 'stop_loss_pct': 0.1, 't..."
